# Recommender systems
### Jimmy Azar

## Introduction

We'll build a simple recommender system. In particular, we explore user-based and item-based collaborative filtering.

## Explore the data

Our data consists of real purchases made between 01.12.2010 and 09.12.2011 at a retail company that sells online gift items.

In [1]:
# load the data
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt

path_to_file = '../data/invoices.csv'
data = pd.read_csv(path_to_file)
print(data.head())
data.shape

  invoice_number stock_code                          description  quantity  \
0         536365     85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1         536365      71053                  WHITE METAL LANTERN         6   
2         536365     84406B       CREAM CUPID HEARTS COAT HANGER         8   
3         536365     84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4         536365     84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

          invoice_date  unit_price  customer_id         country  
0  2010-12-01 08:26:00        2.55      17850.0  United Kingdom  
1  2010-12-01 08:26:00        3.39      17850.0  United Kingdom  
2  2010-12-01 08:26:00        2.75      17850.0  United Kingdom  
3  2010-12-01 08:26:00        3.39      17850.0  United Kingdom  
4  2010-12-01 08:26:00        3.39      17850.0  United Kingdom  


(541909, 8)

## Data preparation

By inspecting the "quantity" field, we notice that there are any quantities that aren't positive. Negative values signify that the order was canceled, and these transactions can be ignored for our purpose.

Next, we inspect the field "customer_id". Notably, there are missing values. So, we remove any observation containing NaNs.

In [2]:
# data preparation
print(data.loc[data['quantity']<0].head(6))
df = data.loc[data['quantity']>0]
df.loc[df['customer_id'].isna()]
df['customer_id'].isna().sum()
df = df.dropna(subset=['customer_id'])
df.shape

    invoice_number stock_code                       description  quantity  \
141        C536379          D                          Discount        -1   
154        C536383     35004C   SET OF 3 COLOURED  FLYING DUCKS        -1   
235        C536391      22556     PLASTERS IN TIN CIRCUS PARADE       -12   
236        C536391      21984   PACK OF 12 PINK PAISLEY TISSUES       -24   
237        C536391      21983   PACK OF 12 BLUE PAISLEY TISSUES       -24   
238        C536391      21980  PACK OF 12 RED RETROSPOT TISSUES       -24   

            invoice_date  unit_price  customer_id         country  
141  2010-12-01 09:41:00       27.50      14527.0  United Kingdom  
154  2010-12-01 09:49:00        4.65      15311.0  United Kingdom  
235  2010-12-01 10:24:00        1.65      17548.0  United Kingdom  
236  2010-12-01 10:24:00        0.29      17548.0  United Kingdom  
237  2010-12-01 10:24:00        0.29      17548.0  United Kingdom  
238  2010-12-01 10:24:00        0.29      17548.0  U

(397924, 8)

## Customer-Item matrix

Before building a recommender system, we need to set up a customer-item (binary) matrix where each row represents a unique customer and each column represents a unique item. A value of 1 in the matrix at location (i,j) indicates that customer (i) purchased item (j), whereas a 0 indicates that no purchase was made. 

In [3]:
# using: pivot_table()
user_item_matrix = pd.pivot_table(df, index='customer_id', columns='stock_code', values='quantity', aggfunc='sum', fill_value=0)
print(user_item_matrix.shape)

df['stock_code'].nunique()
df['customer_id'].nunique()

user_item_matrix = user_item_matrix.applymap(lambda x: 1 if x > 0 else 0)

user_item_matrix.head(6)

(4339, 3665)


stock_code,10002,10080,10120,10123C,10124A,10124G,10125,10133,10135,11001,...,90214V,90214W,90214Y,90214Z,BANK CHARGES,C2,DOT,M,PADS,POST
customer_id,,,,,,,,,,,,,,,,,,,,,
12346.0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
12347.0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
12348.0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
12349.0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
12350.0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
12352.0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,1


## Collaborative filtering

### User-based collaborative filtering

In order to derive the user-user similarity matrix, we need to compute distances between users based on the cosine similarity measure which is the cosine of the angle between two vectors (users), i.e. rows from the user-item matrix. Note that the output will be an $n \times n$ matrix, where $n$ is the number of unique customers.

In [4]:
# user-based collaborative filtering
from sklearn.metrics.pairwise import cosine_similarity

user_user_matrix = pd.DataFrame(cosine_similarity(user_item_matrix), columns=user_item_matrix.index, index=user_item_matrix.index)

user_user_matrix.head()

customer_id,12346.0,12347.0,12348.0,12349.0,12350.0,12352.0,12353.0,12354.0,12355.0,12356.0,...,18273.0,18274.0,18276.0,18277.0,18278.0,18280.0,18281.0,18282.0,18283.0,18287.0
customer_id,,,,,,,,,,,,,,,,,,,,,
12346.0,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,...,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000
12347.0,0.0,1.000000,0.063022,0.046130,0.047795,0.038484,0.0,0.025876,0.136641,0.094742,...,0.0,0.029709,0.052668,0.0,0.032844,0.062318,0.0,0.113776,0.109364,0.012828
12348.0,0.0,0.063022,1.000000,0.024953,0.051709,0.027756,0.0,0.027995,0.118262,0.146427,...,0.0,0.064282,0.113961,0.0,0.000000,0.000000,0.0,0.000000,0.170905,0.083269
12349.0,0.0,0.046130,0.024953,1.000000,0.056773,0.137137,0.0,0.030737,0.032461,0.144692,...,0.0,0.105868,0.000000,0.0,0.039014,0.000000,0.0,0.067574,0.137124,0.030475
12350.0,0.0,0.047795,0.051709,0.056773,1.000000,0.031575,0.0,0.000000,0.000000,0.033315,...,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.044866,0.000000


### Making recommendations 

With the user-user similarity matrix computed, in addition to the user-item matrix, we are now ready to make recommendations. Suppose we wish to make recommendations for user A with customer ID: 13645. We'll find the IDs of the 10 nearest neighbors, i.e. the most similar users to customer A.

Next, we'll extract which items where bought by user A from the user-item matrix. Similarly, we'll extract item IDs that were bought by the 10 most similar users to A. 

Then, we identify items to recommend to A based on items bought by the 10 most similar users to A, but which A hasn't bought yet. Finally, we use the "description" field to obtain the descriptions of the recommended item IDs.

In [5]:
# making recommendations
customer_id = 13645
neighbors = user_user_matrix.loc[customer_id].sort_values(ascending=False).index[1:10+1]
neighbors = neighbors.tolist()

ind = np.where(user_item_matrix.loc[customer_id,:]==1)
items_bought_by_customer = user_item_matrix.columns[ind].tolist()

ind = user_item_matrix.loc[neighbors].any(axis=0)
items_bought_by_neighbors = user_item_matrix.columns[ind].tolist()

items_to_recommend = list(set(items_bought_by_neighbors) - set(items_bought_by_customer))

items_to_recommend_descriptions = df.loc[df['stock_code'].isin(items_to_recommend), ['stock_code','description']].drop_duplicates().set_index('stock_code')

items_to_recommend_descriptions.sort_values(by=['stock_code'], ascending = True)

,description
stock_code,
10133,COLOURING PENCILS BROWN TUBE
16219,HOUSE SHAPE PENCIL SHARPENER
20832,RED FLOCK LOVE HEART PHOTO FRAME
20984,12 PENCILS TALL TUBE POSY
20996,JAZZ HEARTS ADDRESS BOOK
...,...
85123A,WHITE HANGING HEART T-LIGHT HOLDER
85123A,CREAM HANGING HEART T-LIGHT HOLDER
85125,SMALL ROUND CUT GLASS CANDLESTICK


### Item-based collaborative filtering

We can obtain the item-item similarity matrix using cosine similarity. The input to the function in this case is the transpose of the user-item matrix we created previously, and the output should be an $m \times m$ matrix, where $m$ is the number of unique item IDs (stock codes). Using this similarity matrix, we identify the stock codes of the 10 most similar items to the item with stock code 71053. Finally, we extract the descriptions of the item with stock code 71053 and its 10 most similar items.

In [6]:
# item-based collaborative filtering
item_item_matrix = pd.DataFrame(cosine_similarity(user_item_matrix.T), columns=user_item_matrix.columns, index=user_item_matrix.columns)
item_item_matrix.head()

stock_id = '71053'
similar_items = item_item_matrix[stock_id].sort_values(ascending=False).index[0:10+1]

items_to_recommend_descriptions = df.loc[df['stock_code'].isin(similar_items), ['stock_code','description']].drop_duplicates().set_index('stock_code')
items_to_recommend_descriptions.sort_values(by=['stock_code'], ascending = True)

,description
stock_code,
21340,CLASSIC METAL BIRDCAGE PLANT HOLDER
21733,RED HANGING HEART T-LIGHT HOLDER
22224,WHITE LOVEBIRD LANTERN
22423,REGENCY CAKESTAND 3 TIER
22464,HANGING METAL HEART LANTERN
22465,HANGING METAL STAR LANTERN
22784,LANTERN CREAM GAZEBO
23099,FRENCH CARRIAGE LANTERN
71053,WHITE METAL LANTERN
